# Terrain Erosion — Cross-Widget Reactivity

Two reactive widgets:
1. **Terrain Painter** — draw on a 2D canvas, tune simulation params, run erosion
2. **3D Landscape** — real-time Three.js viewer that updates as you paint

Paint on the canvas → the 3D view updates instantly.

In [ ]:
import numpy as np
import vibe_widget as vw

vw.config(model="google/gemini-3.8-flash")


vibe_widget configured ✓


## Widget 1 — Terrain Painter

A 2D canvas where you can:
- Click / drag to raise terrain
- Right-click / drag to lower terrain
- Adjust brush size and strength
- Hit **Erode** to simulate hydraulic erosion
- Hit **Reset** to start fresh

Outputs a `heightmap` (flat array of 64×64 = 4096 floats, values 0–1).

In [2]:
painter = vw.create("""2D terrain painter widget with simulation controls.

    CANVAS:
    - 512x512 px canvas. The terrain grid is 64x64 cells.
    - Left-click / drag raises terrain at the brush position (left-button).
    - Right-click / drag lowers terrain (right-button, suppress context menu).
    - Render each cell with this color gradient interpolated in CSS:
        h=0.0  -> rgb(0,20,120)
        h=0.3  -> rgb(34,139,34)
        h=0.6  -> rgb(180,180,100)
        h=1.0  -> rgb(255,255,255)
    - Redraw the canvas after every brush stroke.

    CONTROLS (rendered below the canvas):
    - Brush Size slider (1-20, default 5, label "Brush Size")
    - Strength slider (0.01-0.2 step 0.01, default 0.05, label "Strength")
    - "Reset" button: reinitialize to procedural hills (same formula as startup), redraw, sync.
    - "Erode" button: run 1500 hydraulic erosion drops, then redraw and sync.
      Erosion loop per drop:
        start at random (x,y); for up to 40 steps:
          scan 8-connected neighbours for the lowest height;
          if current cell is the local min: add 0.003 (deposit) and break;
          else: subtract 0.008 (erode) from current cell, move to lowest neighbour.
      After all drops, clip every value to [0, 1].

    HEIGHTMAP STATE:
    - Maintain heightmap as a plain JS Array of 4096 numbers (64x64 row-major, values 0-1).
    - INITIAL STATE: generate rolling hills using this formula for cell (x, y):
        h = 0.30
          + 0.22 * Math.sin(x / 9)  * Math.sin(y / 9)
          + 0.12 * Math.sin(x / 4 + 1.2) * Math.cos(y / 5)
          + 0.08 * Math.cos(x / 6 + 0.5) * Math.sin(y / 7 + 0.8)
        clamp h to [0, 1]
      This produces a realistic hilly landscape visible from the start.
    - SYNCING: whenever the heightmap changes (after any brush stroke, after Erode,
      after Reset) call:
        model.set('heightmap', heightmap.slice());
        model.save_changes();
      Use Array.from() / .slice() — never pass a Float32Array to model.set().
    - On mount, call syncToModel() once so the linked 3D widget sees the initial hills.
    """,
    outputs=vw.outputs(
        heightmap="Plain JS Array of 4096 numbers (64x64 row-major) in range [0,1]"
    ),
    cache=True,
)
painter

## Widget 2 — 3D Landscape Viewer

A Three.js viewer that reads the `heightmap` from the painter and renders:
- A displaced terrain mesh (mountains / valleys)
- A semi-transparent water plane
- Directional lighting with shadows
- Orbit controls (drag to rotate, scroll to zoom)

The view updates automatically whenever you paint or erode.

In [3]:
landscape = vw.create(
    """
    3D terrain viewer using Three.js.

    IMPORTS — use esm.sh so all bare specifiers are resolved before bundling:
        import * as THREE from 'https://esm.sh/three@0.160.1';
        import { OrbitControls } from 'https://esm.sh/three@0.160.1/examples/jsm/controls/OrbitControls';
    Do NOT use jsdelivr. Do NOT use an import map. Use the full esm.sh URLs directly.

    SETUP:
    - Create a WebGLRenderer (antialias=true) sized to fill the container.
      Use ResizeObserver to call renderer.setSize(w, h) and update camera.aspect.
    - PerspectiveCamera fov=55, near=0.1, far=2000.
      Position at (0, 45, 90), lookAt (0, 5, 0).
    - AmbientLight (0xffffff, 0.5) + DirectionalLight (0xfff4e0, 1.2) at (60, 100, 40).
    - OrbitControls with enableDamping=true.

    TERRAIN MESH:
    - PlaneGeometry(100, 100, 63, 63) rotated -Math.PI/2 around X.
    - MeshStandardMaterial with vertexColors=true, roughness=0.85.
    - updateTerrain(hm) function:
        hm is a plain Array or Array-like of 4096 values.
        If hm is null/undefined/empty, generate fallback hills:
          h = 0.30 + 0.22*Math.sin(i%64/9)*Math.sin(Math.floor(i/64)/9)
              + 0.12*Math.sin((i%64)/4+1.2)*Math.cos(Math.floor(i/64)/5)
        For each vertex i:
          posAttr.setY(i, h * 40)
          color by height band (create new Float32BufferAttribute per call):
            h < 0.12  -> #1a3a6b  (deep water)
            h < 0.18  -> #2d6a9f  (shallow)
            h < 0.40  -> #3a7d44  (lowland)
            h < 0.60  -> #6b8e23  (highland)
            h < 0.80  -> #8b7355  (rock)
            else      -> #f0f0f0  (snow)
        geometry.setAttribute('color', colorAttr)
        posAttr.needsUpdate = true
        geometry.computeVertexNormals()

    WATER PLANE:
    - PlaneGeometry(104, 104) at Y=6, rotation.x=-Math.PI/2.
    - MeshStandardMaterial color=0x006994, transparent=true, opacity=0.6, side=THREE.DoubleSide.

    REACTIVITY:
    - On mount call updateTerrain(model.get('heightmap')).
    - model.on('change:heightmap', () => updateTerrain(model.get('heightmap'))).
    - requestAnimationFrame loop: controls.update(); renderer.render(scene, camera).

    INITIAL HEIGHT: if model.get('heightmap') is null use the same fallback hill formula.
    """,
    inputs=vw.inputs(
        heightmap=painter.outputs.heightmap
    ),
)

In [4]:
import ipywidgets as widgets
from IPython.display import display

GRID = 64

def run_python_erosion(btn):
    btn.disabled = True
    btn.description = "Eroding…"

    # Read current heightmap from the painter widget trait
    raw = painter.outputs.heightmap.value
    if raw is None:
        grid = np.full((GRID, GRID), 0.1, dtype=np.float32)
    else:
        grid = np.array(raw, dtype=np.float32).reshape((GRID, GRID))

    # Hydraulic erosion simulation
    rng = np.random.default_rng()
    for _ in range(3000):
        x, y = rng.integers(0, GRID, 2)
        for __ in range(40):
            best_x, best_y, best_h = x, y, grid[y, x]
            for dy in (-1, 0, 1):
                for dx in (-1, 0, 1):
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < GRID and 0 <= ny < GRID and grid[ny, nx] < best_h:
                        best_h = grid[ny, nx]
                        best_x, best_y = nx, ny
            if best_x == x and best_y == y:
                grid[y, x] = min(1.0, grid[y, x] + 0.003)
                break
            grid[y, x] = max(0.0, grid[y, x] - 0.008)
            x, y = best_x, best_y

    grid = np.clip(grid, 0.0, 1.0)

    # Push result to painter trait — landscape updates automatically
    painter.outputs.heightmap.value = grid.flatten().tolist()

    btn.description = "Run Python Erosion (3 000 drops)"
    btn.disabled = False


erode_btn = widgets.Button(
    description="Run Python Erosion (3 000 drops)",
    layout=widgets.Layout(width="320px", height="40px"),
    button_style="info",
)
erode_btn.on_click(run_python_erosion)

display(widgets.VBox([
    widgets.Label("Push heightmap from Python — both widgets update:"),
    erode_btn,
]))

---
### How cross-widget reactivity works

```python
# Widget A declares an output
painter = vw.create(
    "...",
    outputs=vw.outputs(heightmap="description")
)

# Widget B references that output as an input
landscape = vw.create(
    "...",
    inputs=vw.inputs(
        heightmap=painter.outputs.heightmap   # ExportHandle
    ),
)
```

Vibe Widget:
1. Creates a traitlet on the `painter` widget for `heightmap`.
2. Links it to the `landscape` widget via traitlets observe.
3. Whenever the painter's JS sets `model.set('heightmap', [...])`,
   the landscape widget receives the new value and re-renders its 3D mesh.

You can also push values from Python:
```python
painter.outputs.heightmap.value = my_array.tolist()
```